# Detector event matching (sel_all only)

Match events **before any selection** across detector-variation samples by
`(E / nuE, run, subrun, evt)` and write per-file `*_matched.df`.

Always use **`sel_all`** inputs (raw `evt` / `trk` / `hdr`). Do **not** match at
`sel_mup` / `sel_2prong` — that intersects post-selection survivors and drops
differential efficiency.

Backends: `dent_match_common_events.py` (sel_all), optional SCE script.

Downstream notebooks walk the selection pipeline on these matched files:
`wiremod.ipynb`, `dent.ipynb` → `systematics-detector.ipynb`.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
import sys
from pathlib import Path

REPO = Path("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
sys.path.insert(0, str(REPO))

from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE, SPRING_GEN1_ROOT
from analysis_village.numucc_1p0pi.syst_detvar_common import log, run_dent_match

DFS = Path(os.environ.get("NUMUCC_SPRING_GEN1_ROOT", SPRING_GEN1_ROOT))
OUT_CACHE = Path(os.environ.get(
    "DETECTOR_MATCH_CACHE",
    str(PLOTS_BASE / "systematics-final" / "DetectorMatch" / "cache"),
))
OUT_CACHE.mkdir(parents=True, exist_ok=True)

RUN_WIREMOD_MATCH = True
RUN_DENT_MATCH = True
MAX_FILES = None  # e.g. 3 for a smoke test

print("DFS =", DFS)
print("OUT_CACHE =", OUT_CACHE)


## WireMod match (`sel_all` + calo)

Intersect YZ / XTXW / CV event keys at **sel_all**, then write `*_matched.df`.

Use **sel_all + updatecalo** productions (`evt_cv` / `evt_ccal_*` / … plus `trk`).
Matching preserves calo universe tables for the WireMod envelope walk.


In [ ]:
# sel_all WireMod / CV directories (override as needed)
WIREMOD_VARIATIONS = {
    # Geometry + calo should be remade as sel_all (see make_pandora_evtdf_all_updatecalo).
    # Placeholders — replace with your sel_all WireMod campaign paths:
    "yz": str(DFS / os.environ.get("WIREMOD_YZ_SEL_ALL", "SET_ME__sel_all-mc-WireModYZ")),
    "xtxw": str(DFS / os.environ.get("WIREMOD_XTXW_SEL_ALL", "SET_ME__sel_all-mc-WireModXTXW")),
    "cv": str(DFS / os.environ.get("WIREMOD_CV_SEL_ALL", "SET_ME__sel_all-mc-calovar")),
}

WIREMOD_VARIATIONS = {k: v for k, v in WIREMOD_VARIATIONS.items() if Path(v).is_dir()}
print("WireMod sel_all dirs:", WIREMOD_VARIATIONS)

if RUN_WIREMOD_MATCH and len(WIREMOD_VARIATIONS) >= 2:
    log("WireMod sel_all match …")
    rc = run_dent_match(
        WIREMOD_VARIATIONS,
        fmt="sel_all",
        filename_str="sel_all",
        summary_csv=str(OUT_CACHE / "wiremod_matched_summary-sel_all.csv"),
        max_files=MAX_FILES,
    )
    print("wiremod match exit:", rc)
elif RUN_WIREMOD_MATCH:
    print("skip WireMod match: need ≥2 existing sel_all variation dirs")
else:
    print("skip WireMod match")


## DENT match (`sel_all`)

Same key definition (`hdr` + generator `evt.mc.E`).


In [ ]:
DENT_VARIATIONS = {
    "cv": str(DFS / "2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV_updated"),
    "dent": str(DFS / "2026_09_09_230442__sel_all-mc-BNB_cosmics-detvar_DENT"),
}
# Fallback: Sep-3 DENT_updated if Sep-9 highstats dir missing
_alt_old = {
    "cv": DFS / "2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV_updated",
    "dent": DFS / "2026_09_03_032705__sel_all-mc-BNB_cosmics-detvar_DENT_updated",
}
if not all(Path(v).is_dir() for v in DENT_VARIATIONS.values()) and all(
    p.is_dir() for p in _alt_old.values()
):
    DENT_VARIATIONS = {k: str(v) for k, v in _alt_old.items()}

DENT_MATCHED_OUT = Path(PLOTS_BASE) / "systematics-final" / "DENT-highstats" / "matched"
DENT_MATCHED_SUFFIX = "_matched_hs"

print("DENT sel_all dirs:", DENT_VARIATIONS)
print("matched out:", DENT_MATCHED_OUT, "suffix:", DENT_MATCHED_SUFFIX)

if RUN_DENT_MATCH:
    log("DENT sel_all match (high-stats) …")
    DENT_MATCHED_OUT.mkdir(parents=True, exist_ok=True)
    rc = run_dent_match(
        DENT_VARIATIONS,
        fmt="sel_all",
        filename_str="sel_all",
        summary_csv=str(OUT_CACHE / "dent_matched_summary-sel_all-hs.csv"),
        matched_out_dir=str(DENT_MATCHED_OUT),
        matched_suffix=DENT_MATCHED_SUFFIX,
        max_files=MAX_FILES,
    )
    print("dent match exit:", rc)
else:
    print("skip DENT match")


## Optional SCE

Still `scripts/sce_match_common_events.py` — prefer sel_all inputs there too.
Detector Product B uses **WireMod + DENT**, not SCE.


In [ ]:
print("Matching complete (sel_all). Next: wiremod.ipynb / dent.ipynb (pipeline walk on matched files).")
